In [73]:
from openff.toolkit import Molecule, ForceField

# RDKit doesn't have strong opinions on what charge metals should have
mol = Molecule.from_smiles("C[Zn]C")
#mol = Molecule.from_smiles("C[Zn+1]C")
#mol = Molecule.from_smiles("C[Zn+2]C")

# Or what total bond order to expect
#mol = Molecule.from_smiles("C=[Zn+2]=C")

# dative bonds are A-OK (or maybe they get sanitized to single?)
#mol = Molecule.from_smiles("[CH3-1]->[Zn+1]C")

mol.generate_conformers()
mol

[10:38:45] UFFTYPER: Unrecognized atom type: Zn1+2 (1)


NGLWidget()

In [74]:
import numpy as np 
from openff.units import Quantity
ff = ForceField("openff-2.2.1.offxml")
na = mol.n_atoms

mapped_smiles = mol.to_smiles(mapped=True)
dummy_charges = Quantity(np.arange(-.1, .10001, .2/(mol.n_atoms-1)), "elementary_charge") + (mol.total_charge/mol.n_atoms)

ff["LibraryCharges"].add_parameter({"smirks": mapped_smiles,
                                    "charge": dummy_charges}
                                  )
ff["Bonds"].add_parameter({"smirks": "[Zn:1]~[C:2]",
                           "length": Quantity(2, "angstrom"),
                           "k": Quantity(500, "kcal / mol / angstrom**2")
                          })
ff["Angles"].add_parameter({"smirks": "[C:1]~[Zn:2]~[C:3]",
                           "angle": Quantity(180, "degree"),
                           "k": Quantity(100, "kilocalorie_per_mole /radian ** 2")
                          })
ff["ProperTorsions"].add_parameter({"smirks": "[*:1]~[Zn:2]~[*:3]~[*:4]",
                                    "phase1": Quantity(0, "degree"),
                                    "periodicity1": 1,
                                    "k1": Quantity(0, "kilocalorie_per_mole")
                          })
ff["vdW"].add_parameter({"smirks": "[Zn:1]",
                           "epsilon": Quantity(.1, "kilocalorie / mol"),
                           "rmin_half": Quantity(2, "angstrom")
                          })


In [75]:
interchange = ff.create_interchange(mol.to_topology())
interchange.minimize()
interchange.visualize()

NGLWidget()

In [76]:
import openmm
import openmm.unit as openmm_unit
# Construct and configure a Langevin integrator at 300 K with an appropriate friction constant and time-step
integrator = openmm.LangevinIntegrator(
    300 * openmm_unit.kelvin,
    1 / openmm_unit.picosecond,
    0.002 * openmm_unit.picoseconds,
)

# Under the hood, this creates *OpenMM* `System` and `Topology` objects, then combines them together
simulation = interchange.to_openmm_simulation(integrator=integrator)

stride = 100

dcd_reporter = openmm.app.DCDReporter(file="trajectory.dcd", reportInterval=stride)
simulation.reporters.append(dcd_reporter)

In [77]:
simulation.context.setVelocitiesToTemperature(300 * openmm_unit.kelvin)
simulation.runForClockTime(1 * openmm_unit.second)

In [78]:
import mdtraj
import nglview
trajectory: mdtraj.Trajectory = mdtraj.load(
    "trajectory.dcd", top=mdtraj.Topology.from_openmm(interchange.to_openmm_topology())
)

view = nglview.show_mdtraj(trajectory)
view

NGLWidget(max_frame=53)